<a href="https://colab.research.google.com/github/piaoruilin/hallucination-medical-vlm/blob/main/llava_pmcvqa_qlora_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA fine-tuning of LLaVA-1.5-7B on PMC-VQA (Colab)
Runtime > GPU. Trains a small LoRA adapter (the base model stays frozen), open-ended style:
question -> answer text, with the same prompt as the SLAKE zero-shot baseline.
Checkpoints go to Google Drive, so you can resume after a disconnect.

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00


## 1. Settings and data

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile

repo = "xmcmic/PMC-VQA"
csv_file = hf_hub_download(repo, "train_2.csv", repo_type="dataset")
zip_file = hf_hub_download(repo, "images_2.zip", repo_type="dataset")

with zipfile.ZipFile(zip_file) as z:
    z.extractall("/content/pmcvqa_images")
print(csv_file)

train_2.csv: reconstructing file:   0%|          |  0.00B / 56.7MB            

train_2.csv: downloading bytes:           |  0.00B            

images_2.zip: reconstructing file:   0%|          |  0.00B / 2.21GB            

images_2.zip: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--xmcmic--PMC-VQA/snapshots/b56ae594f794867893143b337b4118a835794647/train_2.csv


In [ ]:
import pandas as pd, glob, os
df = pd.read_csv(csv_file)
print(df.columns.tolist())
print(df.iloc[0])

name = df["Figure_path"].iloc[0]
print(glob.glob(f"/content/pmcvqa_images/**/{os.path.basename(name)}", recursive=True))

['index', 'Figure_path', 'Caption', 'Question', 'Choice A', 'Choice B', 'Choice C', 'Choice D', 'Answer', 'split']
index                                                         35
Figure_path                               PMC8253797_Fig4_11.jpg
Caption        A slightly altered cell . (c-c‴) A highly alte...
Question        What color is used to label the Golgi complex...
Choice A                                               A: Green 
Choice B                                                 B: Red 
Choice C                                          C: Light blue 
Choice D                                               D: Yellow
Answer                                                         B
split                                                      train
Name: 0, dtype: object
['/content/pmcvqa_images/figures/PMC8253797_Fig4_11.jpg']


In [ ]:
import os, re, glob, zipfile, random, torch, pandas as pd
from PIL import Image
from google.colab import drive
from huggingface_hub import hf_hub_download
drive.mount("/content/drive")

# ---- SETTINGS ----
REPO = "xmcmic/PMC-VQA"
EXTRACT_DIR = "/content/pmcvqa_images"
IMG_COL, Q_COL = "Figure_path", "Question"
N_TRAIN = 2000           # start small; scale up once you know the speed
USE_CHEST_FILTER = False # rough keyword filter on the question text
OUT_DIR = "/content/drive/MyDrive/llava_pmcvqa_lora"
SEED = 0
# ------------------

# 1. Download (cached, so rerunning is quick)
CSV_PATH = hf_hub_download(REPO, "train_2.csv", repo_type="dataset")
ZIP_PATH = hf_hub_download(REPO, "images_2.zip", repo_type="dataset")

# 2. Unzip once
if not (os.path.isdir(EXTRACT_DIR) and os.listdir(EXTRACT_DIR)):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

# 3. Load the CSV
df = pd.read_csv(CSV_PATH)
print(df.columns.tolist(), len(df))

# 4. Find the answer column (the README spells it "Anwser")
A_COL = next(c for c in df.columns if c.lower() in ("answer", "anwser"))

# 5. Find the image folder by locating one real image
IMG_DIR = None
for rel in df[IMG_COL].astype(str).head(50):
    hit = next(glob.iglob(f"{EXTRACT_DIR}/**/{os.path.basename(rel)}", recursive=True), None)
    if hit:
        IMG_DIR = hit[:-len(rel)].rstrip("/") if hit.endswith(rel) else os.path.dirname(hit)
        break
assert IMG_DIR, "None of the first 50 images were found. images_2.zip may not match train_2.csv."

print("A_COL   =", A_COL)
print("IMG_DIR =", IMG_DIR)
print(df.iloc[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['index', 'Figure_path', 'Caption', 'Question', 'Choice A', 'Choice B', 'Choice C', 'Choice D', 'Answer', 'split'] 152603
A_COL   = Answer
IMG_DIR = /content/pmcvqa_images/figures
index                                                         35
Figure_path                               PMC8253797_Fig4_11.jpg
Caption        A slightly altered cell . (c-c‴) A highly alte...
Question        What color is used to label the Golgi complex...
Choice A                                               A: Green 
Choice B                                                 B: Red 
Choice C                                          C: Light blue 
Choice D                                               D: Yellow
Answer                                                         B
split                                                      train
Name: 0, dtype: object


In [ ]:
# map letter -> choice column, tolerant of names like "Choice A" / "Choice_A"
choice_col = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}

def get_answer(row):
    a = str(row[A_COL]).strip()
    if re.fullmatch(r"[A-Da-d]", a):                       # it's just a letter
        col = choice_col.get("choice" + a.lower())
        if col and pd.notna(row[col]):
            a = str(row[col]).strip()                      # use the option's text instead
    return re.sub(r"^[A-D][\.\:\)]\s*", "", a)             # drop a leading "B:" / "B."

df["ans"] = df.apply(get_answer, axis=1)

if USE_CHEST_FILTER:
    kw = ["chest", "lung", "thora", "pulmon", "pleur", "mediastin"]
    df = df[df[Q_COL].astype(str).str.lower().str.contains("|".join(kw))]
    print(len(df), "rows after chest filter")

# keep only rows whose image file exists
df = df[df[IMG_COL].apply(lambda p: os.path.exists(os.path.join(IMG_DIR, str(p))))]
print(len(df), "rows with an image on disk")

df = df.sample(n=min(N_TRAIN, len(df)), random_state=SEED).reset_index(drop=True)
print("training on", len(df))
print((df["ans"].str.len() <= 2).mean(), "= fraction of answers that are 1-2 characters (should be near 0)")
df[[Q_COL, "ans"]].head(5)

2000 rows with an image on disk
training on 2000
0.0075 = fraction of answers that are 1-2 characters (should be near 0)


,Question,ans
0,What do the arrows indicate in the images?,Differentiated cells
1,What does the red arrow in the image indicate?,Location of the tumor after treatment
2,What structures are indicated by the arrow in...,Endoplasmic reticulum.
3,What is the type of MRI used in the image?,T1-weighted MRI
4,Which bone was preserved during the procedure?,Fibula


## 2. Model + LoRA

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "llava-hf/llava-1.5-7b-hf"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)   # T4 has no bf16
model = LlavaForConditionalGeneration.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id)

model = prepare_model_for_kbit_training(model)   # also turns on gradient checkpointing
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    # regex: attention layers of the language model only (vision tower stays frozen)
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj)",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.config.use_cache = False
model.print_trainable_parameters()

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

trainable params: 16,777,216 || all params: 7,080,204,288 || trainable%: 0.2370


## 3. Dataset
Prompt is identical to the SLAKE baseline. The loss is computed only on the answer tokens:
everything up to and including `ASSISTANT:` is masked with -100.

In [ ]:
PROMPT = "USER: <image>\n{q}\nAnswer the question using a single word or phrase. ASSISTANT:"

class PMCDataset(torch.utils.data.Dataset):
    def __init__(self, d): self.d = d
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = Image.open(os.path.join(IMG_DIR, str(r[IMG_COL]))).convert("RGB")
        prompt = PROMPT.format(q=r[Q_COL])
        full = prompt + " " + r["ans"] + processor.tokenizer.eos_token
        enc = processor(text=full, images=img, return_tensors="pt")
        plen = processor(text=prompt, images=img, return_tensors="pt")["input_ids"].shape[1]
        labels = enc["input_ids"].clone()
        labels[:, :plen] = -100
        return {"input_ids": enc["input_ids"][0], "attention_mask": enc["attention_mask"][0],
                "pixel_values": enc["pixel_values"][0], "labels": labels[0]}

def collate(batch):                      # batch size is 1, so no padding is needed
    assert len(batch) == 1
    return {k: v.unsqueeze(0) for k, v in batch[0].items()}

train_ds = PMCDataset(df)
x = train_ds[0]
print({k: tuple(v.shape) for k, v in x.items()})
print("answer tokens being trained on:", int((x["labels"] != -100).sum()))

{'input_ids': (616,), 'attention_mask': (616,), 'pixel_values': (3, 336, 336), 'labels': (616,)}
answer tokens being trained on: 6


In [ ]:
x = train_ds[0]
trained = x["input_ids"][x["labels"] != -100]
print(repr(processor.tokenizer.decode(trained)))
print(df.loc[0, [Q_COL, "ans"]])

'Differentiated cells</s>'
Question     What do the arrows indicate in the images? 
ans                                 Differentiated cells
Name: 0, dtype: object


## 4. Train
Effective batch size = 1 x 16 (gradient accumulation). Time the first ~20 steps before committing to a long run.
If you hit an out-of-memory error, restart the session and lower N_TRAIN or check nothing else is using the GPU.

In [ ]:
import math
from transformers import Trainer, TrainingArguments

EPOCHS = 1
GRAD_ACCUM = 16
total_steps = math.ceil(len(train_ds) / GRAD_ACCUM) * EPOCHS
warmup = max(1, int(0.03 * total_steps))          # about 3% of training
print("optimizer steps:", total_steps, "| warmup steps:", warmup)

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)

has_ckpt = os.path.isdir(OUT_DIR) and any(d.startswith("checkpoint-") for d in os.listdir(OUT_DIR))
trainer.train(resume_from_checkpoint=has_ckpt)

optimizer steps: 125 | warmup steps: 3


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,2.539435
20,2.139495


KeyboardInterrupt: 

## 5. Save the adapter

In [ ]:
model.save_pretrained(OUT_DIR + "/final")
processor.save_pretrained(OUT_DIR + "/final")
print("saved to", OUT_DIR + "/final")

## 6. Evaluate on SLAKE (same pipeline as the baseline)
Open your baseline notebook and make two changes, then run everything else **unchanged**:

1. Right after the cell that loads LLaVA, add:
```python
from peft import PeftModel
model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/llava_pmcvqa_lora/final")
model.eval()
```
2. In the save cell, rename the output file (e.g. `finetuned_preds.csv`) so you don't overwrite the baseline predictions.

Use `N = None` for the full test set. Then compare each split side by side:
yes/no accuracy, choice-type accuracy, open recall, open exact match.